<a href="https://colab.research.google.com/github/Santosh-S321/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Santosh-S321/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
os.chdir("/content")
REPO_URL = "https://github.com/Santosh-S321/flyrank-ml-internship"
REPO_DIR = "/content/flyrank-ml-internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

## 1. Two paper findings + my methodology questions

**Finding A — "Refreshing Pages Actually Works" (Refresh ROI, held-out test):** claims
7 of 9 strata show statistically significant refresh lift, including a 52x impression
lift for refreshed 365+ pages.

**My methodology question:** the paper's own Method section names "content age
confounds model comparisons" as a limitation — but doesn't specify whether refreshed
pages were randomly assigned or selected by editors (who likely pick already-promising
candidates to refresh). If selection wasn't random, some of the lift may reflect which
pages got chosen, not the refresh itself. Asked respectfully: was refresh assignment
independent of a page's prior trajectory, or could a page already showing early
recovery signals have been more likely to get refreshed?

**Finding B — Growth Prediction model:** 90% accuracy on same-brand held-out pages,
dropping to 75% on entirely unseen brands.

**My methodology question:** this is the same pattern I tested on my own model in
ML-05 — a random split can let a model memorize brand-specific quirks, and a grouped
split reveals the honest gap. The paper's 15-point drop (90%→75%) is much larger than
my own 2.6-point drop (0.911→0.885 AUC), which raises a genuine question: was
"same-brand, new pages" a true forward-looking holdout (new pages published after the
training window), or a random row-holdout within brand where same-period pages could
still share hidden signal? The size of the drop suggests real memorization was
happening — worth knowing which kind of split produced the 90% number.

In [2]:
print("See markdown above.")

See markdown above.


## 2. My model under an honest split (before/after)

Re-running my Week-5 Random Forest model twice on the same data and features: once
with a plain random split (the "before" — what a less careful analysis might report),
once with the grouped-by-client split I actually used (the "after"). Same metric,
same K, same seed.

In [3]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

candidate_features = ["content_age_days", "days_since_last_update", "impressions_90d",
                       "avg_position", "ctr", "word_count", "engagement_rate", "scroll_rate"]
features = [c for c in candidate_features if c in df.columns]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]

# BEFORE: plain random split
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                    class_weight="balanced_subsample", random_state=42, n_jobs=-1)
rf_random.fit(Xr_tr, yr_tr)
scores_random = rf_random.predict_proba(Xr_te)[:, 1]
p20_random = precision_at_k(scores_random, yr_te.values, 20)
p50_random = precision_at_k(scores_random, yr_te.values, 50)

# AFTER: grouped-by-client split
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
Xg_tr, Xg_te = X.iloc[train_idx], X.iloc[test_idx]
yg_tr, yg_te = y.iloc[train_idx], y.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                     class_weight="balanced_subsample", random_state=42, n_jobs=-1)
rf_grouped.fit(Xg_tr, yg_tr)
scores_grouped = rf_grouped.predict_proba(Xg_te)[:, 1]
p20_grouped = precision_at_k(scores_grouped, yg_te.values, 20)
p50_grouped = precision_at_k(scores_grouped, yg_te.values, 50)

comparison = pd.DataFrame([
    {"split": "BEFORE (random)", "precision_at_20": p20_random, "precision_at_50": p50_random, "base_rate": y.mean()},
    {"split": "AFTER (grouped by client)", "precision_at_20": p20_grouped, "precision_at_50": p50_grouped, "base_rate": y.mean()},
])
print(comparison.round(3))

                       split  precision_at_20  precision_at_50  base_rate
0            BEFORE (random)             1.00             0.94      0.542
1  AFTER (grouped by client)             0.45             0.60      0.542


**Before/after finding:** The random split reported near-perfect precision (1.00 @20,
0.94 @50) — numbers that look excellent but are misleading. The grouped-by-client
split dropped sharply to 0.45 @20 and 0.60 @50, with 0.45 actually falling *below* the
base rate (0.542) — worse than random guessing at the very top of the list. This is a
dramatic memorization signal: the random split let the model learn client-specific
patterns rather than genuinely predictive ones. This mirrors — at an even larger scale
— the paper's own 90%→75% same-brand vs. unseen-brand drop in its Growth Prediction
model (Finding B above). Both cases show the same lesson: a headline number from a
random split can significantly overstate real-world skill, and the honest number only
appears once you group by the entity that repeats.

## 3. Leakage audit

Re-running the same leakage hunt from Week 3/5 on my final Week-5 feature set:
`content_age_days`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`,
`word_count`, `engagement_rate`, `scroll_rate`.

In [4]:
label_source_cols = {"trend_direction", "trend_pct"}
leak_risk = [c for c in features if c in label_source_cols]
print("Label-derived columns in final feature set:", leak_risk if leak_risk else "None — clean")

product_flag_cols = {"health_score", "priority_score", "action_type", "refresh_tier"}
flag_risk = [c for c in features if c in product_flag_cols]
print("Product-flag columns in final feature set:", flag_risk if flag_risk else "None — clean")

# Sanity check: is any single feature suspiciously dominant?
importances = pd.Series(rf_grouped.feature_importances_, index=features).sort_values(ascending=False)
print("\nFeature importances (grouped-split model):")
print(importances.round(3))

Label-derived columns in final feature set: None — clean
Product-flag columns in final feature set: None — clean

Feature importances (grouped-split model):
impressions_90d           0.300
avg_position              0.210
content_age_days          0.180
word_count                0.114
scroll_rate               0.064
ctr                       0.056
days_since_last_update    0.055
engagement_rate           0.020
dtype: float64


## 4. Claim rewrite

**Original claim (from my ML-08 notebook):** "Logistic Regression wins clearly at both
K (0.75 @20, 0.66 @50), beating both the baseline and Random Forest."

**Rewritten in safe language:** On this one grouped-by-client split, using the current
proxy label (`trend_direction`-derived, not an observed future outcome), Logistic
Regression outperformed both the baseline rule and Random Forest at Precision@20 and
Precision@50. This is a directional, single-split observation, not a claim that
Logistic Regression is generally superior for this task — a different split or a
future-window label could change the ranking. Decision-support only: this ranks
candidates for human review, it does not predict that any individual page will
actually decline.

In [5]:
print("Claim rewrite complete — see markdown above.")

Claim rewrite complete — see markdown above.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.